### Cell 1: Import các thư viện cần thiết
Nhập các thư viện cần thiết cho xử lý dữ liệu, tiền xử lý và huấn luyện mô hình.

In [1]:
import os
import sys
import gc
import warnings
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Cấu hình cảnh báo và hiển thị pandas
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', None)
print('Import các thư viện thành công!')

Import các thư viện thành công!


### Cell 2: Đọc dữ liệu từ toàn bộ các năm (2016-2024)
Nạp dữ liệu từ thư mục phân vùng parquet. Hỗ trợ tự động nhận diện đường dẫn từ cả thư mục gốc và thư mục `src/notebooks`.

In [2]:
# Xác định đường dẫn dữ liệu tối ưu và hỗ trợ cả khi chạy từ repo root hoặc từ src/notebooks
candidate_paths = [
    'data/processed/inbound_atl',
    '../../data/processed/inbound_atl',
    '../data/processed/inbound_atl',
    'data/processed/tabular_by_year',
    '../../data/processed/tabular_by_year',
]

data_path = None
for p in candidate_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError('Không tìm thấy thư mục dữ liệu (inbound_atl hoặc tabular_by_year)!')

print(f'Đang tải dữ liệu từ: {data_path} ...')
df = pd.read_parquet(data_path)

print(f'Kích thước tập dữ liệu gốc: {df.shape}')
print('Phân phối dữ liệu theo năm:')
print(df['source_year'].value_counts().sort_index())

Đang tải dữ liệu từ: ../../data/processed/inbound_atl ...


Kích thước tập dữ liệu gốc: (3022433, 38)
Phân phối dữ liệu theo năm:
source_year
2016    381166
2017    358263
2018    386580
2019    391075
2020    242121
2021    309621
2022    311701
2023    332741
2024    309165
Name: count, dtype: int64


### Cell 3: Feature Engineering chung & Xử lý Missing Values
Thực hiện xử lý thời gian và điền dữ liệu khuyết cho toàn bộ tập dữ liệu trước.

In [3]:
# 1. Chuyển đổi thời gian thành giờ
df['CRS_DEP_HOUR'] = pd.to_datetime(df['CRS_DEP_TIME'], errors='coerce').dt.hour
df['CRS_ARR_HOUR'] = pd.to_datetime(df['CRS_ARR_TIME'], errors='coerce').dt.hour

# 2. Xử lý missing values cho thời tiết bằng trung vị (median)
weather_cols = ['O_TEMP', 'O_PRCP', 'O_WSPD', 'D_TEMP', 'D_PRCP', 'D_WSPD']
imputer = SimpleImputer(strategy='median')
df[weather_cols] = imputer.fit_transform(df[weather_cols])

# 3. Label Encoding cho biến Categorical
categorical_cols = ['OP_CARRIER', 'ORIGIN', 'DEST']
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# Loại bỏ NaN ở các cột còn lại
df = df.dropna(subset=['CRS_ELAPSED_TIME', 'CRS_DEP_HOUR', 'CRS_ARR_HOUR', 'ARR_DELAY', 'DEP_DELAY'])

print(f'Kích thước dữ liệu sau khi tiền xử lý chung: {df.shape}')
print(f'Số lượng missing values thời tiết còn lại: {df[weather_cols].isna().sum().sum()}')

Kích thước dữ liệu sau khi tiền xử lý chung: (3022433, 40)
Số lượng missing values thời tiết còn lại: 0


### Cell 4: Hàm chuẩn bị dữ liệu và huấn luyện mô hình
Hàm này cho phép tạo 2 mô hình khác nhau (dự đoán DEP_DELAY và ARR_DELAY) với cấu hình biến (features) tùy chỉnh.

In [4]:
def prepare_and_train(df_full, target_col, exclude_cols, drop_weather_dest=False, model_types=['xgb', 'lr', 'rf']):
    print(f"{'='*50}")
    print(f"BẮT ĐẦU HUẤN LUYỆN CÁC MÔ HÌNH DỰ ĐOÁN: {target_col} >= 15 phút")
    print(f"CÁC MÔ HÌNH ĐƯỢC CHỌN: {model_types}")
    print(f"{'='*50}")
    
    df_model = df_full.copy()
    
    target_name = f'TARGET_{target_col}'
    df_model[target_name] = (df_model[target_col] >= 15).astype(int)
    
    cols_to_drop = exclude_cols + ['FL_DATE', 'flight_key', 'OP_CARRIER_FL_NUM', 'source_row_number', 'FLIGHTS', 'CRS_DEP_TIME', 'CRS_ARR_TIME', 'ARR_DELAY', 'DEP_DELAY']
    
    if drop_weather_dest:
        print("-> Đang loại bỏ các trường thời tiết ở nơi đến (D_TEMP, D_PRCP, D_WSPD)...")
        cols_to_drop += ['D_TEMP', 'D_PRCP', 'D_WSPD', 'D_LATITUDE', 'D_LONGITUDE', 'DEST_INDEX']
        
    df_model = df_model.drop(columns=cols_to_drop, errors='ignore')
    
    train_mask = df_model['source_year'].between(2016, 2022)
    valid_mask = df_model['source_year'] == 2023
    test_mask = df_model['source_year'] == 2024
    
    X_train = df_model[train_mask].drop(columns=[target_name, 'source_year']).copy()
    y_train = df_model[train_mask][target_name].copy()
    
    X_valid = df_model[valid_mask].drop(columns=[target_name, 'source_year']).copy()
    y_valid = df_model[valid_mask][target_name].copy()
    
    X_test = df_model[test_mask].drop(columns=[target_name, 'source_year']).copy()
    y_test = df_model[test_mask][target_name].copy()
    
    print(f"Kích thước Train (2016-2022): {X_train.shape}")
    print(f"Kích thước Valid (2023)    : {X_valid.shape}")
    print(f"Kích thước Test (2024)     : {X_test.shape}")
    
    del df_model
    gc.collect()
    
    numeric_cols = [col for col in X_train.columns if col not in ['OP_CARRIER', 'ORIGIN', 'DEST', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK']]
    scaler = StandardScaler()
    
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_valid[numeric_cols] = scaler.transform(X_valid[numeric_cols])
    X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])
    
    # Chuẩn hóa kiểu dữ liệu số cho tương thích tốt nhất
    for c in X_train.columns:
        if str(X_train[c].dtype).startswith('Int') or str(X_train[c].dtype).startswith('int'):
            X_train[c] = X_train[c].astype(np.int32)
            X_valid[c] = X_valid[c].astype(np.int32)
            X_test[c] = X_test[c].astype(np.int32)
        elif str(X_train[c].dtype).startswith('Float') or str(X_train[c].dtype).startswith('float'):
            X_train[c] = X_train[c].astype(np.float32)
            X_valid[c] = X_valid[c].astype(np.float32)
            X_test[c] = X_test[c].astype(np.float32)
            
    trained_models = {}
    
    for m_type in model_types:
        print(f"{'-'*40}")
        print(f"Đang huấn luyện mô hình {m_type.upper()}...")
        if m_type == 'rf':
            clf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
        elif m_type == 'xgb':
            try:
                import xgboost as xgb
                try:
                    clf = xgb.XGBClassifier(n_estimators=100, max_depth=10, random_state=42, tree_method='hist', device='cuda')
                    clf.fit(X_train.iloc[:5], y_train.iloc[:5])
                except Exception:
                    print("-> GPU không khả dụng cho XGBoost, chuyển sang sử dụng CPU...")
                    clf = xgb.XGBClassifier(n_estimators=100, max_depth=10, random_state=42, tree_method='hist', device='cpu', n_jobs=-1)
                clf = xgb.XGBClassifier(n_estimators=100, max_depth=10, random_state=42, tree_method='hist', device='cuda')
            except ImportError:
                from sklearn.ensemble import HistGradientBoostingClassifier
                print("-> Không tìm thấy XGBoost, sử dụng HistGradientBoostingClassifier...")
                clf = HistGradientBoostingClassifier(max_iter=100, max_depth=10, random_state=42)
        elif m_type == 'lr':
            try:
                from cuml.linear_model import LogisticRegression as cuLogisticRegression
                print("Sử dụng cuML Logistic Regression (GPU)...")
                clf = cuLogisticRegression(max_iter=200)
            except (ImportError, Exception):
                from sklearn.linear_model import LogisticRegression
                print("Không tìm thấy cuML, fallback về sklearn Logistic Regression (CPU)...")
                clf = LogisticRegression(max_iter=200)
        else:
            print(f"Bỏ qua {m_type}: model_type không hợp lệ.")
            continue
        
        clf.fit(X_train, y_train)
        
        print(f"Đang đánh giá {m_type.upper()} trên tập Validation (2023)...")
        y_val_pred = clf.predict(X_valid)
        y_val_prob = clf.predict_proba(X_valid)[:, 1]
        print(f"Validation ROC-AUC: {roc_auc_score(y_valid, y_val_prob):.4f}")
        
        print(f"Đang dự đoán và đánh giá {m_type.upper()} trên tập Test (2024)...")
        y_test_pred = clf.predict(X_test)
        y_test_prob = clf.predict_proba(X_test)[:, 1]
        
        print(f"--- Báo cáo kết quả phân loại {m_type.upper()} (Test 2024) ---")
        print(classification_report(y_test, y_test_pred))
        print(f"Test ROC-AUC Score: {roc_auc_score(y_test, y_test_prob):.4f}")
        
        trained_models[m_type] = clf
        
    return trained_models

### Cell 5: Mô hình 1 - Dự đoán Khởi hành trễ (DEP_DELAY >= 15)
Đối với dự đoán khởi hành trễ, chúng ta không biết các thông tin diễn ra trong và sau khi bay (như Taxi Out/In, Wheels Off/On).

In [5]:
# Các cột không được sử dụng khi dự đoán Khởi hành trễ
leakage_dep = [
    'DEP_TIME', 'TAXI_OUT', 'WHEELS_OFF', 
    'WHEELS_ON', 'TAXI_IN', 'ARR_TIME', 
    'ACTUAL_ELAPSED_TIME', 'AIR_TIME'
]

models_dep = prepare_and_train(
    df, 
    target_col='DEP_DELAY', 
    exclude_cols=leakage_dep, 
    drop_weather_dest=False,
    model_types=['xgb', 'lr', 'rf']
)

BẮT ĐẦU HUẤN LUYỆN CÁC MÔ HÌNH DỰ ĐOÁN: DEP_DELAY >= 15 phút
CÁC MÔ HÌNH ĐƯỢC CHỌN: ['xgb', 'lr', 'rf']


Kích thước Train (2016-2022): (2380527, 22)
Kích thước Valid (2023)    : (332741, 22)
Kích thước Test (2024)     : (309165, 22)


----------------------------------------
Đang huấn luyện mô hình XGB...


Đang đánh giá XGB trên tập Validation (2023)...


Validation ROC-AUC: 0.6916
Đang dự đoán và đánh giá XGB trên tập Test (2024)...
--- Báo cáo kết quả phân loại XGB (Test 2024) ---


...

--- Báo cáo kết quả phân loại LR (Test 2024) ---
              precision    recall  f1-score   support

           0       0.82      1.00      0.90    254658
           1       0.58      0.00      0.00     54507

    accuracy                           0.82    309165
   macro avg       0.70      0.50      0.45    309165
weighted avg       0.78      0.82      0.74    309165

Test ROC-AUC Score: 0.6610
----------------------------------------
Đang huấn luyện mô hình RF...


Đang đánh giá RF trên tập Validation (2023)...


Validation ROC-AUC: 0.6932
Đang dự đoán và đánh giá RF trên tập Test (2024)...


--- Báo cáo kết quả phân loại RF (Test 2024) ---
              precision    recall  f1-score   support

           0       0.82      1.00      0.90    254658
           1       0.58      0.00      0.01     54507

    accuracy                           0.82    309165
   macro avg       0.70      0.50      0.46    309165
weighted avg       0.78      0.82      0.75    309165



Test ROC-AUC Score: 0.6893


### Cell 6: Mô hình 2 - Dự đoán Đến trễ (ARR_DELAY >= 15)
Đối với dự đoán đến trễ, **nếu** dự đoán được thực hiện trước khi cất cánh, chúng ta không thể dùng DEP_DELAY. **Tuy nhiên**, nếu bài toán là dự đoán sau khi đã cất cánh, ta có thể đưa DEP_DELAY vào.\nTheo yêu cầu của bạn: Không sử dụng các trường thời tiết ở nơi đến (`D_*`) để dự đoán.

In [6]:
# Các cột rò rỉ dữ liệu cho Arrival Delay (nếu giả sử dự đoán từ lúc trước khi bay)
# Nếu bạn muốn dùng DEP_DELAY làm biến dự đoán cho ARR_DELAY (tức là dự đoán trên không), hãy bỏ 'DEP_DELAY' khỏi danh sách này.
leakage_arr = [
    'DEP_TIME', 'TAXI_OUT', 'WHEELS_OFF', 
    'WHEELS_ON', 'TAXI_IN', 'ARR_TIME', 
    'ACTUAL_ELAPSED_TIME', 'AIR_TIME'
]

models_arr = prepare_and_train(
    df, 
    target_col='ARR_DELAY', 
    exclude_cols=leakage_arr, 
    drop_weather_dest=True,  # Loại bỏ D_TEMP, D_PRCP, D_WSPD theo yêu cầu
    model_types=['xgb', 'lr', 'rf']
)

BẮT ĐẦU HUẤN LUYỆN CÁC MÔ HÌNH DỰ ĐOÁN: ARR_DELAY >= 15 phút
CÁC MÔ HÌNH ĐƯỢC CHỌN: ['xgb', 'lr', 'rf']


-> Đang loại bỏ các trường thời tiết ở nơi đến (D_TEMP, D_PRCP, D_WSPD)...


Kích thước Train (2016-2022): (2380527, 16)
Kích thước Valid (2023)    : (332741, 16)
Kích thước Test (2024)     : (309165, 16)


----------------------------------------
Đang huấn luyện mô hình XGB...


Đang đánh giá XGB trên tập Validation (2023)...


...

Đang đánh giá LR trên tập Validation (2023)...
Validation ROC-AUC: 0.6416
Đang dự đoán và đánh giá LR trên tập Test (2024)...
--- Báo cáo kết quả phân loại LR (Test 2024) ---


              precision    recall  f1-score   support

           0       0.83      1.00      0.90    255127
           1       0.00      0.00      0.00     54038

    accuracy                           0.83    309165
   macro avg       0.41      0.50      0.45    309165
weighted avg       0.68      0.83      0.75    309165

Test ROC-AUC Score: 0.6325
----------------------------------------
Đang huấn luyện mô hình RF...


Đang đánh giá RF trên tập Validation (2023)...


Validation ROC-AUC: 0.6900
Đang dự đoán và đánh giá RF trên tập Test (2024)...


--- Báo cáo kết quả phân loại RF (Test 2024) ---
              precision    recall  f1-score   support

           0       0.83      1.00      0.90    255127
           1       0.63      0.00      0.01     54038

    accuracy                           0.83    309165
   macro avg       0.73      0.50      0.46    309165
weighted avg       0.79      0.83      0.75    309165

Test ROC-AUC Score: 0.6826
